In [3]:
print("Let's Start")

Let's Start


In [4]:
from sklearn.datasets import load_iris

x,y = load_iris(return_X_y=True , as_frame=True)
x.shape ,y.shape

((150, 4), (150,))

In [5]:
from sklearn.model_selection import train_test_split
x_train , x_text , y_train , y_text = train_test_split(x , y , test_size=0.2 , random_state=0 )

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
lr = LogisticRegression()
lr.fit(x_train , y_train)
lr_pred = lr.predict(x_text)

accuracy_score(y_text , lr_pred)

1.0

My logistic class with OVR method

In [ ]:
import numpy as np


class MyLogisticRegression:

    def __init__(self, iterations=100, lr_=0.1):
        if iterations <= 0:
            raise ValueError("iterations must be greater than 0.")

        if lr_ <= 0:
            raise ValueError("lr_ must be greater than 0.")

        self.coef_ = None
        self.intercept_ = None
        self.coefs= None
        self.intercepts =None
        self.iterations = iterations
        self.lr_ = lr_

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def partical_fit(self, x, y):

        x = np.asarray(x, dtype=float)
        y = np.asarray(y)

        if x.ndim != 2:
            raise ValueError(
                "X must be a 2-dimensional array."
            )

        if y.ndim != 1:
            y = y.ravel()

        if x.shape[0] != y.shape[0]:
            raise ValueError(
                "X and y must have the same number of samples."
            )

        if len(np.unique(y)) != 2:
            raise ValueError(
                "MyLogisticRegression supports binary classification only."
            )

        if not np.all(np.isin(y, [0, 1])):
            raise ValueError(
                "For binary classification, y must contain only 0 and 1."
            )

        y = y.astype(float)

        n_samples, n_features = x.shape

        weights = np.zeros(n_features)
        intercept = 0.0

        for _ in range(self.iterations):

            # Linear equation
            z = np.dot(x, weights) + intercept

            # Probability
            y_hat = self._sigmoid(z)

            # Gradient
            error = y_hat - y

            dw = np.dot(x.T, error) / n_samples
            db = np.sum(error) / n_samples

            # Gradient descent
            weights -= self.lr_ * dw
            intercept -= self.lr_ * db

        print("Model Trained Successfully")
        self.coef_ = weights
        self.intercept_ = intercept

        return self.coef_ , self.intercept_

    def fit(self , x, y):
        from sklearn.preprocessing import OneHotEncoder
        ohe = OneHotEncoder()
        y_ohe = ohe.fit_transform(y.values.reshape(-1 ,1))
        y_ohe = y_ohe.toarray()
        coefs ={}
        intercepts = {}
        for i in range(y_ohe.shape[1]):
            self.partical_fit(x , y_ohe[:,i])
            coefs[f'Model {i}'] = self.coef_
            intercepts[f'Model {i}'] = self.intercept_
        self.coefs = coefs
        self.intercepts = intercepts
        
    def predict_proba(self, x):

        if self.coef_ is None:
            raise ValueError(
                "Model must be fitted before prediction."
            )

        x = np.asarray(x, dtype=float)

        if x.ndim != 2:
            raise ValueError(
                "X must be a 2-dimensional array."
            )

        if x.shape[1] != len(self.coef_):
            raise ValueError(
                f"Expected {len(self.coef_)} features, "
                f"got {x.shape[1]}."
            )

        
        z = np.dot(x, self.coef_) + self.intercept_

        return self._sigmoid(z)


    def predict(self, x, threshold=0.5):

        if not 0 < threshold < 1:
            raise ValueError(
                "threshold must be between 0 and 1."
            )

        probabilities = self.predict_proba(x)

        return (probabilities >= threshold).astype(int)

In [8]:
my = MyLogisticRegression()
my.fit(x_train , y_train)

Model Trained Successfully
Model Trained Successfully
Model Trained Successfully


In [21]:
it = my.intercepts
c = my.coefs
c

{'Model 0': array([ 0.2814298 ,  0.89498638, -1.47065952, -0.67499753]),
 'Model 1': array([-0.06492069, -0.51960434,  0.29910237, -0.11716245]),
 'Model 2': array([-0.66073924, -0.74757812,  1.12968089,  0.7880298 ])}

In [22]:
probs = {}
def _sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))
for i in range(len(c)):
    z = np.dot(x,c[f'Model {i}']) + it[f'Model {i}']
    probs[f'Model {i}'] = _sigmoid(z)




In [62]:
scaled_probs = {}
p1 = []
p2 = []
p3 = []

In [63]:
for i in range(len(x)):
    d1= probs['Model 0'][i]/(probs['Model 0'][i] + probs['Model 1'][i] + probs['Model 2'][i])
    p1.append(d1)
scaled_probs['Model 0'] = p1

In [64]:
for i in range(len(x)):
    d2= probs['Model 1'][i]/(probs['Model 0'][i] + probs['Model 1'][i] + probs['Model 2'][i])
    p2.append(d2)
scaled_probs['Model 1'] = p2

In [65]:
for i in range(len(x)):
    d3= probs['Model 2'][i]/(probs['Model 0'][i] + probs['Model 1'][i] + probs['Model 2'][i])
    p3.append(d3)
scaled_probs['Model 2']= p3

In [68]:
scaled_probs['Model 0'][0]

np.float64(0.8495585873974433)

In [69]:
scaled_probs['Model 1'][0]

np.float64(0.1410300662731328)

In [71]:
scaled_probs['Model 2'][0]

np.float64(0.00941134632942385)

In [37]:
p = p1 + p2 + p3
p

np.float64(1.0)

In [25]:
predicts = {}
threshold = 0.5
for i in range(len(probs)):
    probabilities = probs[f'Model {i}']

    predicts[f'Model {i}'] = (probabilities >= threshold).astype(int)

In [26]:
predicts

{'Model 0': array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'Model 1': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 

In [38]:
probs

{'Model 0': array([9.27438527e-01, 8.85361593e-01, 9.10029119e-01, 8.70142241e-01,
        9.31460787e-01, 9.17905131e-01, 9.04674608e-01, 9.07483202e-01,
        8.59844582e-01, 8.86365848e-01, 9.34890118e-01, 8.88937411e-01,
        8.89295987e-01, 9.15603153e-01, 9.70304613e-01, 9.62321993e-01,
        9.52686145e-01, 9.22762917e-01, 9.22488499e-01, 9.30985213e-01,
        8.91070712e-01, 9.20189476e-01, 9.56271422e-01, 8.48809996e-01,
        8.37362744e-01, 8.55477468e-01, 8.80925045e-01, 9.19017303e-01,
        9.23199680e-01, 8.66781082e-01, 8.59535460e-01, 9.05582953e-01,
        9.54067479e-01, 9.63982034e-01, 8.79387842e-01, 9.27269635e-01,
        9.43086760e-01, 9.33931119e-01, 8.86004627e-01, 9.09819074e-01,
        9.30823961e-01, 7.99747514e-01, 9.02872920e-01, 8.76068849e-01,
        8.75027221e-01, 8.75290688e-01, 9.25694302e-01, 8.94615796e-01,
        9.33155928e-01, 9.12204356e-01, 5.47350824e-02, 5.78023257e-02,
        3.46152084e-02, 4.83545888e-02, 3.66817807e-0